In [11]:
import numpy as np
import igl
import meshplot as mp
import trimesh
from plyfile import PlyData
from collections import defaultdict
from utils.mesh_utils import *


In [12]:
# show each patch from .ply file
def segmentation_from_ply_face_color(ply_path: str, viz=False):
    ply = PlyData.read(ply_path)

    if "face" not in ply:
        raise ValueError("PLY 中没有 face element。")

    vertice_data = ply["vertex"].data
    face_data = ply["face"].data
    
    v = np.stack([vertice_data["x"], vertice_data["y"], vertice_data["z"]], axis=1).astype(np.float64)
    f = np.stack(face_data["vertex_indices"], axis=0).astype(np.int64)

    # 1) 取三角面顶点索引
    if "vertex_indices" not in face_data.dtype.names:
        raise ValueError("face element 中没有 vertex_indices 字段。")
    faces = np.vstack(face_data["vertex_indices"]).astype(np.int64)
    if faces.shape[1] != 3:
        raise ValueError(f"当前 faces 不是三角形 (shape={faces.shape})，需要先三角化或改代码处理多边形。")

    # 2) 取 face 颜色字段（兼容不同命名）
    names = set(face_data.dtype.names)
    candidates = [
        ("red", "green", "blue"),
        ("r", "g", "b"),
        ("diffuse_red", "diffuse_green", "diffuse_blue"),
    ]
    rgb_fields = None
    for c in candidates:
        if all(k in names for k in c):
            rgb_fields = c
            break
    if rgb_fields is None:
        raise ValueError(
            f"face element 没找到颜色字段，现有字段: {face_data.dtype.names}\n"
            "如果颜色在 vertex 上，用方案B。"
        )

    colors = np.stack([face_data[rgb_fields[0]], face_data[rgb_fields[1]], face_data[rgb_fields[2]]], axis=1).astype(np.uint8)

    # 3) 颜色 -> label（相同 RGB 视为同类）
    # np.unique 会按行去重并返回 inverse 映射
    unique_colors, labels = np.unique(colors, axis=0, return_inverse=True)
    
    if viz:
    # show each patch
        for i in range(unique_colors.shape[0]):
            patch_f = f[labels==i]
            print(f'patch {i}: #f: {patch_f.shape[0]}')
            f_colors = np.ones((f.shape[0], 3))
            f_colors[np.where(labels==i)] = [1, 0, 0]
            
            mp.plot(v, f, f_colors, shading={'wireframe': False})
        
    v, f = np.array(v), np.array(f)
    return v, f, labels


def _normalize(v, eps=1e-3):
    n = np.linalg.norm(v, axis=-1, keepdims=True)
    return v / np.maximum(n, eps)

def _tri_normals(V, F):
    a = V[F[:, 0]]
    b = V[F[:, 1]]
    c = V[F[:, 2]]
    n = np.cross(b - a, c - a)
    return _normalize(n)

def _tri_centroids(V, F):
    return (V[F[:, 0]] + V[F[:, 1]] + V[F[:, 2]]) / 3.0

def _tri_aabbs(V, F):
    pts = V[F]  # (m,3,3)
    mn = pts.min(axis=1)
    mx = pts.max(axis=1)
    return mn, mx

def _point_in_tri_barycentric(p, a, b, c, eps=1e-3):
    """
    3D barycentric test in triangle plane.
    Return True if p is inside triangle abc (including boundary).
    """
    v0 = b - a
    v1 = c - a
    v2 = p - a

    d00 = np.dot(v0, v0)
    d01 = np.dot(v0, v1)
    d11 = np.dot(v1, v1)
    d20 = np.dot(v2, v0)
    d21 = np.dot(v2, v1)

    denom = d00 * d11 - d01 * d01
    if abs(denom) < 1e-30:
        return False  # degenerate

    v = (d11 * d20 - d01 * d21) / denom
    w = (d00 * d21 - d01 * d20) / denom
    u = 1.0 - v - w

    # allow boundary with eps
    return (u >= -eps) and (v >= -eps) and (w >= -eps)

def map_faces_coarse_to_refined(
    v1, f1, v2, f2,
    eps_plane_rel=1e-3,
    eps_aabb_rel=1e-3,
    eps_ang=1e-3,
    bary_eps=1e-3,
    allow_flip_normal=True,
):
    """
    输入:
      v1: (n1,3) float
      f1: (m1,3) int  原始三角面
      v2: (n2,3) float
      f2: (m2,3) int  细化后模型三角面

    输出:
      coarse_to_fine: list[list[int]]  长度 m1，每个原始面对应的细化面索引列表
      fine_to_coarse: (m2,) int        每个细化面对应的原始面索引；找不到则为 -1
    """
    v1 = np.asarray(v1, dtype=np.float64)
    f1 = np.asarray(f1, dtype=np.int64)
    v2 = np.asarray(v2, dtype=np.float64)
    f2 = np.asarray(f2, dtype=np.int64)

    # scale-based tolerances
    allv = np.vstack([v1, v2])
    bbox_min = allv.min(axis=0)
    bbox_max = allv.max(axis=0)
    diag = np.linalg.norm(bbox_max - bbox_min)
    if diag == 0:
        diag = 1.0

    eps_plane = eps_plane_rel * diag
    eps_aabb = eps_aabb_rel * diag

    # Precompute coarse face data
    n1 = _tri_normals(v1, f1)                       # (m1,3)
    a1 = v1[f1[:, 0]]                               # (m1,3)
    d1 = -np.einsum("ij,ij->i", n1, a1)             # plane: n·x + d = 0
    mn1, mx1 = _tri_aabbs(v1, f1)                   # AABB for quick reject

    # Precompute refined face data
    c2 = _tri_centroids(v2, f2)                     # (m2,3)
    n2 = _tri_normals(v2, f2)                       # (m2,3)

    fine_to_coarse = np.full((f2.shape[0],), -1, dtype=np.int64)

    # Main loop: assign each refined face to a coarse face
    for j in range(f2.shape[0]):
        p = c2[j]
        nn2 = n2[j]

        # AABB candidate filter: p must lie in coarse face AABB (with tolerance)
        cand = np.where(
            (p[0] >= mn1[:, 0] - eps_aabb) & (p[0] <= mx1[:, 0] + eps_aabb) &
            (p[1] >= mn1[:, 1] - eps_aabb) & (p[1] <= mx1[:, 1] + eps_aabb) &
            (p[2] >= mn1[:, 2] - eps_aabb) & (p[2] <= mx1[:, 2] + eps_aabb)
        )[0]

        if cand.size == 0:
            continue

        best_i = -1
        best_score = -np.inf

        for i in cand:
            nn1 = n1[i]

            # normal consistency
            dotn = float(np.dot(nn1, nn2))
            if allow_flip_normal:
                if abs(dotn) < 1.0 - eps_ang:
                    continue
                score = abs(dotn)
            else:
                if dotn < 1.0 - eps_ang:
                    continue
                score = dotn

            # coplanar test: distance to plane
            dist = abs(np.dot(nn1, p) + d1[i])
            if dist > eps_plane:
                continue

            # point-in-triangle test using barycentric
            ia, ib, ic = f1[i]
            A = v1[ia]; B = v1[ib]; C = v1[ic]
            if not _point_in_tri_barycentric(p, A, B, C, eps=bary_eps):
                continue

            # choose best by score, tie-break by smaller plane distance
            # (you can add more tie-break rules if needed)
            score2 = score - 1e-3 * (dist / (eps_plane + 1e-30))
            if score2 > best_score:
                best_score = score2
                best_i = i

        fine_to_coarse[j] = best_i

    # Build inverse mapping
    coarse_to_fine = [[] for _ in range(f1.shape[0])]
    for j, i in enumerate(fine_to_coarse):
        if i >= 0:
            coarse_to_fine[i].append(j)

    return coarse_to_fine, fine_to_coarse



In [13]:
def save_pid(labels, save_path):
    labels = np.asarray(labels).reshape(-1)
    with open(save_path, 'w') as file:
        for l in labels:
            file.write(f'{int(l)}\n')
            
def save_seg(labels, seg_path):
    labels = np.asarray(labels).reshape(-1)
    with open(seg_path, 'w') as file:
        for l in labels:
            file.write(f'{int(l)}\n')

def save_obj(v, f, save_path):
    with open(save_path, 'w') as file:
        for i in range(v.shape[0]):
            file.write(f'v {v[i, 0]} {v[i, 1]} {v[i, 2]}\n')
        for i in range(f.shape[0]):
            file.write(f'f {f[i, 0]+1} {f[i, 1]+1} {f[i, 2]+1}\n')


def show_info(ply_path: str, viz=False, save_to_seg=False, save_to_pid=False, obj_path=None, pid_path=None):
    v, f, labels = segmentation_from_ply_face_color(ply_path, viz)
    print(f'load mesh: {ply_path.split("/")[-1]},\n#v: {v.shape[0]}, #f: {f.shape[0]}',"labels:", labels.shape, "min/max:", labels.min(), labels.max())
    if save_to_pid:
        save_obj(v, f, obj_path)
        save_pid(labels, pid_path)
        print(f'saved to {obj_path} and {pid_path}')
    
def convert_patch(origin_ply_path, split_ply_path):
    v1, f1, labels1 = segmentation_from_ply_face_color(origin_ply_path)
    v2, f2, labels2 = segmentation_from_ply_face_color(split_ply_path)
    # v1, f1 = igl.read_triangle_mesh(origin_ply_path)
    # v2, f2 = igl.read_triangle_mesh(split_ply_path)
    print(v1.shape, f1.shape, v2.shape, f2.shape)
    coarse_to_fine, fine_to_coarse = map_faces_coarse_to_refined(v1, f1, v2, f2)
    
    fine_to_coarse = np.asarray(fine_to_coarse, dtype=np.int64)
    labels1_map = np.full((fine_to_coarse.shape[0], ), -1, dtype=np.int64)
    labels1_map[fine_to_coarse>=0] = labels1[fine_to_coarse[fine_to_coarse>=0]]
    
    print(labels1.shape, labels1_map.shape, labels2.shape)
    
    # print(np.linalg.norm(labels1_map - labels2))
    
    return v1, f1, labels1, v2, f2, labels1_map, labels2
    


def direct_save_pid(ply_path, save_path):
    v, f, labels = segmentation_from_ply_face_color(ply_path)
    save_pid(labels, save_path)
    
def direct_save_obj(ply_path, save_path):
    v, f, labels = segmentation_from_ply_face_color(ply_path)
    save_obj(v, f, save_path)


In [ ]:
# model_name = '2'
# model_name = 'cad16'
# model_name = '00140180_7b0779cf3270ccee61e01b3e_trimesh_000'
# model_name = "00140223_5374ce893909eddbc7536b71_trimesh_000"
# model_name = "00140302_dce0fb9e808b32ded73025c1_trimesh_000"
# model_name = "00140553_e2c0841b6c86e3bfdcc8c477_trimesh_000"
# model_name = "00140750_0da4d13f288d4cd70deecf20_trimesh_001"
# model_name = "00140919_3023c3cfd7d6b1a80700e17e_trimesh_000"
# model_name = "00140936_9a2b76f7e489ea90f5e41ef8_trimesh_007"



# model_name = "00140400_e4a5e7862a9beda824b2e00c_trimesh_003"
# model_name = '00140710_33073a6cb55fadd20ec0e3fb_trimesh_001'
# model_name = "00140756_eaf6606df2a105f356a99677_trimesh_000"
# model_name = "00140923_0e818d7ff8780682567c824e_trimesh_000"
# model_name = "00140967_631426bb5199fa39fc8eeb1c_trimesh_008"
# model_name = "00140982_f2e481e34cff3a450799acac_trimesh_000"
# model_name = "00141097_57222034e4b08229a097b208_trimesh_010"

# model_name = "00023435_385b221a0d58490b84f3edee_trimesh_000"
# model_name = "00023471_7f45ff9e8c754def8bb4b1cb_trimesh_000"
# model_name = "00023576_35c7024f831f44048104c331_trimesh_000"
# model_name = "00023582_9c917172a61b472fb0e6ae3c_trimesh_002"
# model_name = "00023792_30c31f050d2c40139e9b36ca_trimesh_001"
# model_name = "00024036_de448e53313b4faa9dff2245_trimesh_003"
# model_name = "00024040_de448e53313b4faa9dff2245_trimesh_007"
# model_name = "00024082_de448e53313b4faa9dff2245_trimesh_049"

# model_name = "00024132_a087bf84d26046349bbbd6f6_trimesh_000"
# model_name = "00024448_274a7c9d6def4a699961a747_trimesh_002"
# model_name = "00024792_34a17822747a4b20a8c2954b_trimesh_009"
# model_name = "00024961_e26f4b039c6c48c9878b21fe_trimesh_000"
# model_name = "00025063_9347be1a9dad4e70852dcce2_trimesh_000"
# model_name = "00025183_021c990acfc648618a49bd47_trimesh_000"

# TODO
# model_name = '00140316_62dfc752c2876cd94647532a_trimesh_011'
# model_name = "00140682_572115b8e4b01ff83d93b7c0_trimesh_000"
# model_name = '00140708_1c9a6a91c5121e671690124c_trimesh_000'
# model_name = '00140722_0752ced969c82bbb19ce666b_trimesh_000'
# model_name = "00140780_d56b3836d8a40e3035d42380_trimesh_000"
# model_name = "00141000_e5a1548042d09e9e460ec51e_trimesh_000"
# model_name = "00141172_5722364be4b0e6559901e691_trimesh_000"
# model_name = "00141176_5ff893b2e5c70dac030147d6_trimesh_001"

# model_name = "00024044_de448e53313b4faa9dff2245_trimesh_011"
# model_name = "00024089_de448e53313b4faa9dff2245_trimesh_056"


#### method.seg -> .obj.pid
def seg_to_pid(seg_file, pid_file):
    with open(seg_file, 'r') as seg_f:
        lines = seg_f.readlines()
    with open(pid_file, 'w') as pid_f:
        for line in lines:
            proxy_id = int(line.strip())
            pid_f.write(f'{proxy_id}\n')

# for model_name in models_name:
#     seg_path_our = f'example_data/final/selected/{model_name}_our.seg'
#     pid_path_our = f'example_data/final/selected/{model_name}.obj.pid'
#     seg_to_pid(seg_path_our, pid_path_our)


#### .ply -> method.seg
def ply_to_seg(ply_path, seg_path, obj_path=None):
    v, f, labels = segmentation_from_ply_face_color(ply_path)
    save_seg(labels, seg_path)
    save_obj(v, f, obj_path)
    

def seg_to_ply(seg_file, obj_file, ply_path):
    v, f = igl.read_triangle_mesh(obj_file)
    face_num = f.shape[0]
    face_to_proxy = [-1] * face_num
    # print(face_num)
    with open(seg_file, "r") as f_pid:
        for face_id, line in enumerate(f_pid):
            proxy_id = int(line.strip())
            face_to_proxy[face_id] = proxy_id

    face_colors = np.ones((face_num, 3))
    for face_id, proxy_id in enumerate(face_to_proxy):
        face_colors[face_id] = colors[proxy_id]
    print(ply_path)
    write_ply_file(v, f, face_colors, ply_path)


#### .obj.pid -> method.seg
def pid_to_seg(pid_file, seg_path):
    seg_to_pid(pid_file, seg_path)


def obj_to_ply(obj_file, ply_path):
    v, f = igl.read_triangle_mesh(obj_file)
    gray = np.array([0.75, 0.75, 0.75], dtype=np.float32)
    write_ply_file(v, f, np.tile(gray, (f.shape[0], 1)), ply_path)


models_name = [
    # paper
    # '2'
    # '00140180_7b0779cf3270ccee61e01b3e_trimesh_000',
    # "00140223_5374ce893909eddbc7536b71_trimesh_000",
    # "00140302_dce0fb9e808b32ded73025c1_trimesh_000",
    
    # done
    # "00023435_385b221a0d58490b84f3edee_trimesh_000",
    # "00023471_7f45ff9e8c754def8bb4b1cb_trimesh_000",
    # "00023576_35c7024f831f44048104c331_trimesh_000",
    # "00023582_9c917172a61b472fb0e6ae3c_trimesh_002",
    # "00023792_30c31f050d2c40139e9b36ca_trimesh_001",
    # "00024036_de448e53313b4faa9dff2245_trimesh_003",
    # "00024040_de448e53313b4faa9dff2245_trimesh_007",
    # "00024082_de448e53313b4faa9dff2245_trimesh_049",
    # "00024132_a087bf84d26046349bbbd6f6_trimesh_000",
    # "00024448_274a7c9d6def4a699961a747_trimesh_002",
    # "00024792_34a17822747a4b20a8c2954b_trimesh_009",
    # "00024961_e26f4b039c6c48c9878b21fe_trimesh_000",
    # "00025063_9347be1a9dad4e70852dcce2_trimesh_000",
    
    # do not do
    # 'cad16'
    
    
    # todo
    # "00025183_021c990acfc648618a49bd47_trimesh_000",
    # "00025611_37f3a717a9b842cfbe5f6005_trimesh_002",
    # "00025633_2aaf78a0f41a47b1bae8c447_trimesh_003",
    # "00025710_6fe81cf35e2740b3bbde9aa6_trimesh_020",
    # "00025728_6fe81cf35e2740b3bbde9aa6_trimesh_038",
    # "00025749_c01a880ceccb4e05908f8255_trimesh_000",
    # "00026055_11394f3abb83419b88f6eed9_trimesh_000",
    # "00026133_8010d7708c06499e9e99ce1c_trimesh_008",

    # "00140400_e4a5e7862a9beda824b2e00c_trimesh_003",
    # "00140553_e2c0841b6c86e3bfdcc8c477_trimesh_000",
    # '00140710_33073a6cb55fadd20ec0e3fb_trimesh_001',
    # "00140750_0da4d13f288d4cd70deecf20_trimesh_001",
    # "00140756_eaf6606df2a105f356a99677_trimesh_000",
    # "00140919_3023c3cfd7d6b1a80700e17e_trimesh_000",
    # "00140923_0e818d7ff8780682567c824e_trimesh_000",
    # "00140936_9a2b76f7e489ea90f5e41ef8_trimesh_007",
    # "00140967_631426bb5199fa39fc8eeb1c_trimesh_008",
    # "00140982_f2e481e34cff3a450799acac_trimesh_000",
    "00141097_57222034e4b08229a097b208_trimesh_010",
    
]

model_name = models_name[0]
ply_path_our = f"example_results/merge/selected/example_result/our/{model_name}.ply"
ply_path_gt = f"example_results/merge/selected/example_result/gt/{model_name}.ply"
ply_path_raw = f"example_results/merge/selected/example_result/raw/{model_name}.ply"
ply_path_part2surf = f"example_results/merge/selected/example_result/part2surf/{model_name}.ply"
seg_path_our = f'example_data/final/selected/{model_name}_our.seg'
seg_path_gt = f'example_data/final/selected/{model_name}_gt.seg'
seg_path_part2surf = f'example_data/final/selected/{model_name}_part2surf.seg'
pid_file = f'example_data/final/selected/{model_name}.obj.pid'
obj_file = f'example_data/final/selected/{model_name}.obj'


# ply_to_seg(ply_path_part2surf, seg_path_part2surf, obj_file)
# seg_to_pid(seg_path_part2surf, pid_file)

# pid_to_seg(pid_file, seg_path_part2surf)
# seg_to_ply(seg_path_part2surf, obj_file, ply_path_part2surf)

# ply_to_seg(ply_path_our, seg_path_our, obj_file)
# seg_to_pid(seg_path_our, pid_file)

# pid_to_seg(pid_file, seg_path_our)
# seg_to_ply(seg_path_our, obj_file, ply_path_our)


# ply_to_seg(ply_path_gt, seg_path_gt, obj_file)
# seg_to_pid(seg_path_gt, pid_file)

pid_to_seg(pid_file, seg_path_gt)
seg_to_ply(seg_path_gt, obj_file, ply_path_gt)
# obj_to_ply(obj_file, ply_path_raw)

# for model_name in models_name:
#     obj_file = f'example_data/final/selected/{model_name}.obj'
#     ply_path_raw = f"example_results/merge/selected/example_result/raw/{model_name}.ply"
#     obj_to_ply(obj_file, ply_path_raw)

# for model_name in models_name:
#     # seg_path_gt = f'example_data/final/selected/{model_name}_gt.seg'
#     obj_file = f'example_data/final/selected/{model_name}.obj'
#     ply_path_raw = f"example_results/merge/selected/example_result/raw/{model_name}.ply"
#     # ply_path_gt = f"example_results/merge/selected/example_result/gt/{model_name}.ply"
#     # seg_to_ply(seg_path_gt, obj_file, ply_path_gt)
#     obj_to_ply(obj_file, ply_path_raw)

#     pid_file = f'example_data/final/selected/{model_name}.obj.pid'
# obj_file = f'example_data/final/selected/{model_name}.obj'

#     gt_seg_file = f'example_data/final/selected/{model_name}_gt.seg'
#     seg_to_pid(gt_seg_file, pid_file)

# ply_path_our       = f"example_results/merge/selected/example_result/our/{model_name}.ply"
# ply_path_part2surf = f"example_results/merge/selected/example_result/part2surf/{model_name}.ply"
# ply_path_samesh    = f"example_results/merge/selected/example_result/samesh/{model_name}.ply"
# ply_path_segmat    = f"example_results/merge/selected/example_result/segmat/{model_name}.ply"
# ply_path_partfield = f"example_results/merge/selected/example_result/partfield/{model_name}.ply"
# ply_path_partfield_new = f"example_results/merge/selected/example_result/partfield_new/{model_name}.ply"

# seg_path_samesh = f'example_data/final/selected/{model_name}_samesh.seg'
# seg_path_segmat = f'example_data/final/selected/{model_name}_segmat.seg'
# seg_path_partfield = f'example_data/final/selected/{model_name}_partfield.seg'


def converting():
    # show_info(ply_path_our, viz=True)
    show_info(ply_path_our, viz=False)
    # show_info(ply_path_part2surf, viz=False)
    show_info(ply_path_samesh, viz=False)
    show_info(ply_path_partfield, viz=False)
    show_info(ply_path_segmat, viz=False)

# converting()

example_results/merge/selected/example_result/gt/00140923_0e818d7ff8780682567c824e_trimesh_000.ply


In [120]:
# 1. direct save
direct_save_pid(ply_path_segmat, seg_path_segmat)
direct_save_pid(ply_path_samesh, seg_path_samesh)
direct_save_pid(ply_path_partfield, seg_path_partfield)
seg_to_ply(seg_path_segmat, obj_file, ply_path_segmat)
seg_to_ply(seg_path_samesh, obj_file, ply_path_samesh)
seg_to_ply(seg_path_partfield, obj_file, ply_path_partfield)
# seg_to_ply(seg_path_partfield, obj_file, ply_path_partfield_new)

# direct_save_obj(ply_path_our, obj_path)
# direct_save_pid(ply_path_our, pid_path)

example_results/merge/selected/example_result/segmat/00140180_7b0779cf3270ccee61e01b3e_trimesh_000.ply
example_results/merge/selected/example_result/samesh/00140180_7b0779cf3270ccee61e01b3e_trimesh_000.ply
example_results/merge/selected/example_result/partfield/00140180_7b0779cf3270ccee61e01b3e_trimesh_000.ply


In [87]:
# 2. convert and save
    
v1, f1, labels1, v2, f2, labels1_map, labels2 =convert_patch(ply_path_samesh, ply_path_our)
save_pid(labels1_map, seg_path_samesh)
v1, f1, labels1, v2, f2, labels1_map, labels2 =convert_patch(ply_path_partfield, ply_path_our)
save_pid(labels1_map, seg_path_partfield)
v1, f1, labels1, v2, f2, labels1_map, labels2 =convert_patch(ply_path_segmat, ply_path_our)
save_pid(labels1_map, seg_path_segmat)
seg_to_ply(seg_path_samesh, obj_file, ply_path_samesh)
seg_to_ply(seg_path_segmat, obj_file, ply_path_segmat)
seg_to_ply(seg_path_partfield, obj_file, ply_path_partfield_new)

# save_pid(labels2, f'example_data/final/selected/{model_name}_our.seg')
# save_obj(v2, f2, f'example_data/final/selected/{model_name}.obj')
# save_pid(labels2, f'example_data/final/selected/{model_name}.obj.pid')
# save_pid(labels1_map, f'example_data/final/selected/{model_name}.obj.pid')

(71796, 3) (23932, 3) (12026, 3) (24052, 3)
(23932,) (24052,) (24052,)
(11966, 3) (23932, 3) (12026, 3) (24052, 3)
(23932,) (24052,) (24052,)
(71796, 3) (23932, 3) (12026, 3) (24052, 3)
(23932,) (24052,) (24052,)
